# Re-seeded validation: simulation

Runs re-seeded drogued drifter simulations for all 6 drifters. Saves
results to `output/reseeded_results.pkl` for the visualization notebook
(11b).

## Imports

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from parcels import FieldSet, Particle, ParticleSet, StatusCode, Variable
from parcels._core.statuscodes import FieldOutOfBoundError
from parcels.kernels import AdvectionRK4

from drogued_drifters.drifter import DroguedDrifter

## Parameters

In [ ]:
LON_MIN = 9.0
LON_MAX = 13.0
LAT_MIN = 53.5
LAT_MAX = 56.0
TIME_START = "2023-04-24"
TIME_END = "2023-05-10"
DROGUE_DEPTH = 3.0
DT = 300.0
RESEED_INTERVAL_HOURS = 6
LEAD_TIMES_HOURS = [1, 2, 4, 8, 16]
FORECAST_HOURS = 16
RESAMPLE_INTERVAL = "1h"
CSV_PATH = "examples/baltic_drifters/data/drifters_clean.csv"
CMEMS_DIR = "examples/baltic_drifters/data/cmems"

## Load observed trajectories

Resample to hourly fixes so that each drifter has a regular time grid.

In [ ]:
_DEG2M = 1852.0 * 60.0

df_raw = pd.read_csv(CSV_PATH, parse_dates=["date_UTC"])
drifter_ids = sorted(df_raw["D_number"].unique())

# Resample to hourly and interpolate positions
obs = {}
for d_num in drifter_ids:
    grp = df_raw[df_raw["D_number"] == d_num].set_index("date_UTC").sort_index()
    grp_h = grp[["Latitude", "Longitude"]].resample(RESAMPLE_INTERVAL).mean().interpolate()
    grp_h = grp_h.dropna()
    obs[d_num] = grp_h

for d_num, grp in obs.items():
    print(f"  D{d_num}: {len(grp)} hourly fixes, "
          f"{grp.index.min()} to {grp.index.max()}")

## Build effective current field

Same pipeline as notebooks 08 and 09a: CMEMS physics + wave partitions,
Stokes drift profiles at each depth, interpolate onto physics grid,
add to get effective current, build FieldSet.

In [ ]:
LON = slice(LON_MIN, LON_MAX)
LAT = slice(LAT_MIN, LAT_MAX)
TIME = slice(TIME_START, TIME_END)

ds_phy = xr.open_dataset(CMEMS_DIR + "/cmems_mod_bal_phy_anfc_PT1H-i.nc").sel(
    longitude=LON, latitude=LAT, time=TIME,
).load()

ds_phy

In [ ]:
WAVE_VARS = [
    "VHM0_WW", "VTM01_WW", "VMDR_WW",
    "VHM0_SW1", "VTM01_SW1", "VMDR_SW1",
    "VHM0_SW2", "VTM01_SW2", "VMDR_SW2",
]

ds_wav = xr.open_dataset(CMEMS_DIR + "/cmems_mod_bal_wav_anfc_PT1H-i.nc").sel(
    longitude=LON, latitude=LAT, time=TIME,
)[WAVE_VARS].load()

ds_wav

### Stokes drift profiles from wave partitions

In [ ]:
g = 9.81
depth_levels = np.concatenate([[0.0], ds_phy.depth.values])

PARTITIONS = [
    ("VHM0_WW", "VTM01_WW", "VMDR_WW"),
    ("VHM0_SW1", "VTM01_SW1", "VMDR_SW1"),
    ("VHM0_SW2", "VTM01_SW2", "VMDR_SW2"),
]

u_stokes = xr.DataArray(
    np.zeros((len(ds_wav.time), len(depth_levels), len(ds_wav.latitude), len(ds_wav.longitude))),
    dims=["time", "depth", "latitude", "longitude"],
    coords={"time": ds_wav.time, "depth": depth_levels,
            "latitude": ds_wav.latitude, "longitude": ds_wav.longitude},
)
v_stokes = u_stokes.copy()

for hs_var, tp_var, dir_var in PARTITIONS:
    Hs = ds_wav[hs_var]
    T = ds_wav[tp_var]
    dir_from = ds_wav[dir_var]
    valid = Hs.notnull() & T.notnull() & dir_from.notnull() & (T > 0)
    A = Hs / 2.0
    sigma = 2 * np.pi / T
    k = sigma**2 / g
    theta = np.deg2rad(270.0 - dir_from)
    stokes_surf = A**2 * sigma * k

    for i, z in enumerate(depth_levels):
        decay = np.exp(-2 * k * float(z))
        u_stokes[dict(depth=i)] += (stokes_surf * decay * np.cos(theta)).where(valid, 0.0).fillna(0.0)
        v_stokes[dict(depth=i)] += (stokes_surf * decay * np.sin(theta)).where(valid, 0.0).fillna(0.0)

### Interpolate Stokes onto physics grid and build effective current

In [ ]:
ds_stokes = xr.Dataset({"u_stokes": u_stokes, "v_stokes": v_stokes})

ds_stokes_phys = ds_stokes.interp(
    longitude=ds_phy.longitude,
    latitude=ds_phy.latitude,
    method="linear",
).fillna(0.0)

ds_phy_z0 = ds_phy.isel(depth=0).assign_coords(depth=0.0)
ds_phy_ext = xr.concat([ds_phy_z0, ds_phy], dim="depth")

land_mask = ds_phy_ext["uo"].isnull()
ds_stokes_phys["u_stokes"] = ds_stokes_phys["u_stokes"].where(~land_mask)
ds_stokes_phys["v_stokes"] = ds_stokes_phys["v_stokes"].where(~land_mask)

ds_eff = xr.Dataset({
    "U": (ds_phy_ext["uo"] + ds_stokes_phys["u_stokes"]).fillna(0.0),
    "V": (ds_phy_ext["vo"] + ds_stokes_phys["v_stokes"]).fillna(0.0),
})
ds_eff

### Build FieldSet

In [ ]:
ds_eff["grid"] = xr.DataArray(
    data=0,
    attrs={
        "cf_role": "grid_topology",
        "topology_dimension": 2,
        "node_dimensions": "longitude latitude",
        "face_dimensions": (
            "longitude:longitude (padding: none) "
            "latitude:latitude (padding: none)"
        ),
        "vertical_dimensions": "depth:depth (padding: none)",
        "node_coordinates": "longitude latitude",
    },
)

fieldset = FieldSet.from_sgrid_conventions(ds_eff, mesh="spherical")
fieldset.add_constant("drogue_depth", DROGUE_DEPTH)

## Drifter kernel

Same kernel as notebook 09a. Parcels returns velocities in deg/s on a
spherical mesh; the kernel converts to m/s for the Cartesian drifter
model and converts back afterwards. Trajectories are collected in a
Python dict since the Parcels v4 zarr writer has a known bug with
custom kernels.

In [ ]:
DrifterParticle = Particle.add_variable(Variable("z_eff", dtype=np.float64, initial=0.0))

dd = DroguedDrifter()
_warm_state = {}

_traj_store = {"lon": [], "lat": [], "time": [], "pid": []}


def DroguedDrifterKernel(particles, fieldset):
    """Advect particles at the steady-state drift velocity of a buoy+drogue."""
    drogue_depth = fieldset.drogue_depth

    n = len(np.asarray(particles.lon))
    z_surface = np.zeros(n)
    z_drogue = np.full(n, drogue_depth)
    try:
        (u_b, v_b) = fieldset.UV[
            particles.time, z_surface, particles.lat, particles.lon, particles
        ]
        (u_d, v_d) = fieldset.UV[
            particles.time, z_drogue, particles.lat, particles.lon, particles
        ]
    except FieldOutOfBoundError:
        particles.state = StatusCode.Delete
        return

    u_b, v_b = np.asarray(u_b), np.asarray(v_b)
    u_d, v_d = np.asarray(u_d), np.asarray(v_d)
    dt = np.asarray(particles.dt)
    lon_arr = np.asarray(particles.lon)
    lat_arr = np.asarray(particles.lat)
    time_arr = np.asarray(particles.time)
    pid_arr = np.asarray(particles.trajectory)

    cos_lat = np.cos(np.deg2rad(lat_arr))
    deg2m_lon = _DEG2M * cos_lat
    u_b_ms = u_b * deg2m_lon
    v_b_ms = v_b * _DEG2M
    u_d_ms = u_d * deg2m_lon
    v_d_ms = v_d * _DEG2M

    n = len(u_b_ms)
    y0_warm = _warm_state.get("Y") if _warm_state.get("n") == n else None
    xd_drift_ms, yd_drift_ms, theta_final, Y_final = dd.get_final_drift_batch(
        U_b=u_b_ms, V_b=v_b_ms, U_d=u_d_ms, V_d=v_d_ms, y0=y0_warm,
    )
    _warm_state["Y"] = Y_final
    _warm_state["n"] = n
    z_eff = -dd.l * np.cos(theta_final)

    xd_drift = xd_drift_ms / deg2m_lon
    yd_drift = yd_drift_ms / _DEG2M

    particles.dlon += xd_drift * dt
    particles.dlat += yd_drift * dt
    particles.z_eff = z_eff

    _traj_store["lon"].append(lon_arr.copy())
    _traj_store["lat"].append(lat_arr.copy())
    _traj_store["time"].append(time_arr.copy())
    _traj_store["pid"].append(pid_arr.copy())


def DeleteOOB(particles, fieldset):
    """Convert out-of-bounds errors to Delete status."""
    state = np.asarray(particles.state)
    oob = (state == StatusCode.ErrorOutOfBounds) | (state == StatusCode.ErrorThroughSurface)
    if np.any(oob):
        particles.state = np.where(oob, StatusCode.Delete, state)

## Run re-seeded simulations

For each drifter and re-seeding time, deploy three virtual particles:
- Drogued drifter (buoy+drogue model)
- Surface point particle (z=0, AdvectionRK4)
- 3m point particle (z=3m, AdvectionRK4)

Each re-seeding is run as a separate Parcels execution to avoid
warm-state cross-talk.

In [ ]:
SHALLOWEST_DEPTH = float(ds_eff.depth.values[0])
FORECAST_SECONDS = FORECAST_HOURS * 3600
fieldset_time_origin = ds_eff.time.values[0]
fieldset_time_end = np.datetime64(ds_eff.time.values[-1])


def run_pp(fieldset, lon0, lat0, z0, t_start, runtime, dt):
    """Run a single point particle and return (lons, lats, times)."""
    store = {"lon": [], "lat": [], "time": []}

    def CollectKernel(particles, fieldset):
        store["lon"].append(np.asarray(particles.lon).copy())
        store["lat"].append(np.asarray(particles.lat).copy())
        store["time"].append(np.asarray(particles.time).copy())

    pset = ParticleSet(
        fieldset=fieldset, pclass=Particle,
        lon=[lon0], lat=[lat0], z=[z0],
        time=[np.datetime64(t_start)],
    )
    try:
        pset.execute(
            kernels=[AdvectionRK4, CollectKernel, DeleteOOB],
            dt=dt, runtime=runtime, verbose_progress=False,
        )
    except Exception:
        pass
    if not store["lon"]:
        return None
    return (
        np.array([a[0] for a in store["lon"]]),
        np.array([a[0] for a in store["lat"]]),
        np.array([a[0] for a in store["time"]]),
    )


# Collect results: list of dicts with sim_type key
results = []

for d_num in drifter_ids:
    grp = obs[d_num]
    times = grp.index
    reseed_times = times[::RESEED_INTERVAL_HOURS]

    for t_start in reseed_times:
        t_end_target = t_start + pd.Timedelta(hours=FORECAST_HOURS)
        if np.datetime64(t_end_target) > fieldset_time_end:
            continue

        obs_window = grp.loc[t_start:t_end_target]
        if len(obs_window) < 2:
            continue

        lon0 = float(obs_window.iloc[0]["Longitude"])
        lat0 = float(obs_window.iloc[0]["Latitude"])

        # --- Drogued drifter ---
        _warm_state.clear()
        _traj_store["lon"].clear()
        _traj_store["lat"].clear()
        _traj_store["time"].clear()
        _traj_store["pid"].clear()

        pset = ParticleSet(
            fieldset=fieldset, pclass=DrifterParticle,
            lon=[lon0], lat=[lat0], z=[SHALLOWEST_DEPTH],
            time=[np.datetime64(t_start)],
        )
        try:
            pset.execute(
                kernels=[DroguedDrifterKernel, DeleteOOB],
                dt=DT, runtime=FORECAST_SECONDS, verbose_progress=False,
            )
        except Exception as e:
            print(f"  D{d_num} t={t_start} DD: failed ({e})")

        if _traj_store["lon"]:
            results.append({
                "drifter": d_num, "t_start": t_start, "sim_type": "dd",
                "sim_lon": np.array([a[0] for a in _traj_store["lon"]]),
                "sim_lat": np.array([a[0] for a in _traj_store["lat"]]),
                "sim_time": np.array([a[0] for a in _traj_store["time"]]),
            })

        # --- Surface point particle (z=0) ---
        pp = run_pp(fieldset, lon0, lat0, SHALLOWEST_DEPTH, t_start,
                    FORECAST_SECONDS, DT)
        if pp is not None:
            results.append({
                "drifter": d_num, "t_start": t_start, "sim_type": "surface",
                "sim_lon": pp[0], "sim_lat": pp[1], "sim_time": pp[2],
            })

        # --- 3m point particle ---
        pp = run_pp(fieldset, lon0, lat0, DROGUE_DEPTH, t_start,
                    FORECAST_SECONDS, DT)
        if pp is not None:
            results.append({
                "drifter": d_num, "t_start": t_start, "sim_type": "3m",
                "sim_lon": pp[0], "sim_lat": pp[1], "sim_time": pp[2],
            })

    n_dd = sum(1 for r in results if r["drifter"] == d_num and r["sim_type"] == "dd")
    print(f"  D{d_num}: {n_dd} re-seedings (x3 sim types)")

print(f"\nTotal results: {len(results)}")

## Save results

Write simulated and observed trajectories to CSV for the visualization
notebook (11b).

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("examples/baltic_drifters/output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save re-seeded trajectories as CSV
rows = []
for r in results:
    sim_times_dt = pd.DatetimeIndex(
        fieldset_time_origin + (r["sim_time"] * 1e9).astype("timedelta64[ns]")
    )
    for lon, lat, t in zip(r["sim_lon"], r["sim_lat"], sim_times_dt):
        rows.append({
            "drifter": r["drifter"],
            "sim_type": r["sim_type"],
            "reseed_time": r["t_start"],
            "time": t,
            "lon": lon,
            "lat": lat,
        })

df_sim = pd.DataFrame(rows)
csv_path = OUTPUT_DIR / "reseeded_trajectories.csv"
df_sim.to_csv(csv_path, index=False)

# Save observed hourly trajectories too
obs_rows = []
for d_num, grp in obs.items():
    for t, row in grp.iterrows():
        obs_rows.append({
            "drifter": d_num,
            "time": t,
            "lon": row["Longitude"],
            "lat": row["Latitude"],
        })

df_obs = pd.DataFrame(obs_rows)
obs_path = OUTPUT_DIR / "reseeded_obs.csv"
df_obs.to_csv(obs_path, index=False)

for st in ["dd", "surface", "3m"]:
    n = df_sim[df_sim["sim_type"] == st].groupby(["drifter", "reseed_time"]).ngroups
    print(f"  {st}: {n} re-seedings")
print(f"Saved {len(df_sim)} points to {csv_path}")